# Plot the Results of Model Evaluations

In [ ]:
from pathlib import Path

from adjustText import adjust_text
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import numpy as np
import pandas as pd
import seaborn as sns

from ultrabench.models.echocare.echocare_backbone import EchoCareBackbone
from ultrabench.models.samus.samus_backbone import SAMUSBackbone
from ultrabench.models.ultrasam.ultrasam_backbone import UltraSAMBackbone
from ultrabench.models.urfm.urfm_backbone import URFMBackbone
from ultrabench.models.usfm.usfm_backbone import USFMBackbone
from ultrabench.models.usfmae.usfmae_backbone import USFMAEBackbone
from ultrabench.models.deblurmim.deblurmim_backbone import DeblurMIMBackbone
from ultrabench.models.dinov3.dinov3_backbone import (
    DINOv3ViTBackbone,
    DINOv3ConvNeXtBackbone,
)
from ultrabench.models.openus.openus_backbone import OpenUSBackbone
from ultrabench.models.ultrafedfm.ultrafedfm_backbone import UltraFedFMBackbone
from ultrabench.models.tinyusfm.tinyusfm_backbone import TinyUSFMBackbone

MODEL_DISPLAY_NAMES = {
    "deblurmim_convmae": "Deblurring ConvMAE",
    "deblurmim_mae": "Deblurring MAE",
    "dinov3_convnext_base": "DINOv3 ConvNeXt-B",
    "dinov3_convnext_large": "DINOv3 ConvNeXt-L",
    "dinov3_vitb16": "DINOv3 ViT-B",
    "dinov3_vitl16": "DINOv3 ViT-L",
    "echocare": "EchoCare",
    "openus": "OpenUS",
    "samus": "SAMUS",
    "tinyusfm": "TinyUSFM",
    "ultrafedfm": "UltraFedFM",
    "ultrasam": "UltraSAM",
    "urfm_base": "URFM-B",
    "urfm_large": "URFM-L",
    "usfm": "USFM",
    "usfmae": "USF-MAE",
}

# Configure plotting libraries
plt.rcParams["figure.figsize"] = (6.5, 2.48)
sns.set_theme(style="whitegrid", context="paper", palette="colorblind", font_scale=0.8)

# Assign a colour to each model so that coloring is consistent across all
# figures. There are more models than the "colorblind" palette has distinct
# colors, so use "husl" (what seaborn itself falls back to in that case).
MODEL_COLORS = dict(
    zip(
        MODEL_DISPLAY_NAMES.keys(),
        sns.color_palette("husl", n_colors=len(MODEL_DISPLAY_NAMES)),
    )
)

## Load Aggregated Test Results

Load the `test_results.csv` file generated by `aggregate_results.ipynb`.

In [ ]:
test_results = pd.read_csv("test_results_20260826.csv")
test_results.head()

## Performance vs. Release Date

In [ ]:
# The date of the publication of the FIRST version of the paper describing the
# model on arXiv.
release_dates = {
    "deblurmim_convmae": "2023-06-14",
    "deblurmim_mae": "2023-06-14",
    "dinov3_convnext_base": "2025-08-13",
    "dinov3_convnext_large": "2025-08-13",
    "dinov3_vitb16": "2025-08-13",
    "dinov3_vitl16": "2025-08-13",
    "echocare": "2025-09-15",
    "openus": "2025-11-14",
    "samus": "2023-09-13",
    "tinyusfm": "2025-10-22",
    "ultrafedfm": "2024-11-25",
    "ultrasam": "2024-11-25",
    "urfm_base": "2025-08-15",
    "urfm_large": "2025-08-15",
    "usfm": "2023-12-30",
    "usfmae": "2025-10-27",
}

In [ ]:
# Group the classification results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `macro_f1` for each group
classification_test_results = test_results[
    test_results["task"].str.contains("classification")
]

classification_test_summary = (
    classification_test_results.groupby(["task", "model", "shot_pct", "mode"])
    .agg(
        {
            "f1_macro": ["mean"],
        }
    )
    .reset_index()
)

# Flatten the multi-index columns
classification_test_summary.columns = [
    "task",
    "model",
    "shot_pct",
    "mode",
    "macro_f1",
]

classification_test_summary["release_date"] = classification_test_summary["model"].map(
    release_dates
)
classification_test_summary["release_date"] = pd.to_datetime(
    classification_test_summary["release_date"]
)

classification_test_summary.head()

In [ ]:
# Plot overall classification performance by release date
plot_df = (
    classification_test_summary[
        (classification_test_summary["shot_pct"] == 1.0)
        & (classification_test_summary["mode"] == "linear_classifier_full_finetuning")
    ]
    .groupby(["model", "release_date"])
    .agg({"macro_f1": "mean"})
    .reset_index()
)

fig, ax = plt.subplots()

sns.scatterplot(
    data=plot_df,
    x="release_date",
    y="macro_f1",
    hue="model",
    ax=ax,
    legend=False,
)

# Add labels for each point
texts = []
for _, row in plot_df.iterrows():
    texts.append(
        ax.text(
            row["release_date"],
            row["macro_f1"],
            MODEL_DISPLAY_NAMES[row["model"]],
            ha="center",
            va="center",
        )
    )
adjust_text(texts)

# Draw a line connecting the current best model to the next best model
sorted_df = plot_df.sort_values(
    by=["release_date", "macro_f1"], ascending=[True, False]
)
current_best_model = sorted_df.iloc[0]
for i in range(1, len(sorted_df)):
    next_model = sorted_df.iloc[i]
    if next_model["macro_f1"] > current_best_model["macro_f1"]:
        ax.plot(
            [current_best_model["release_date"], next_model["release_date"]],
            [current_best_model["macro_f1"], next_model["macro_f1"]],
            color="gray",
            linestyle="--",
            linewidth=0.5,
        )
        current_best_model = next_model

ax.set_ylabel("Mean F1 Score")
ax.set_ylim(0.5, 0.7)

x_labels = ax.get_xticklabels()
x_ticks = ax.get_xticks()
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels)
ax.set_xlabel("Release Date")

plt.tight_layout()
plt.savefig("classification_performance_by_release_date.pdf")
plt.savefig("classification_performance_by_release_date.png", dpi=300)
plt.show()

In [ ]:
# Group the segmentation results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `positive_dice` for each group.
segmentation_test_results = test_results[
    test_results["task"].str.contains("segmentation")
]

segmentation_test_summary = (
    segmentation_test_results.groupby(["task", "model", "shot_pct", "mode"])
    .agg(
        {
            "positive_dice": ["mean"],
        }
    )
    .reset_index()
)

# Flatten the multi-index columns
segmentation_test_summary.columns = [
    "task",
    "model",
    "shot_pct",
    "mode",
    "positive_dice",
]

segmentation_test_summary["release_date"] = segmentation_test_summary["model"].map(
    release_dates
)
segmentation_test_summary["release_date"] = pd.to_datetime(
    segmentation_test_summary["release_date"]
)

segmentation_test_summary.head()

In [ ]:
# Plot overall segmentation performance by release date
plot_df = (
    segmentation_test_summary[
        (segmentation_test_summary["shot_pct"] == 1.0)
        & (segmentation_test_summary["mode"].str.contains("full_finetuning"))
    ]
    .groupby(["model", "release_date"])
    .agg({"positive_dice": "mean"})
    .reset_index()
)

fig, ax = plt.subplots()

sns.scatterplot(
    data=plot_df,
    x="release_date",
    y="positive_dice",
    hue="model",
    ax=ax,
    legend=False,
)

# Add labels for each point
texts = []
for _, row in plot_df.iterrows():
    texts.append(
        ax.text(
            row["release_date"],
            row["positive_dice"],
            MODEL_DISPLAY_NAMES[row["model"]],
            ha="center",
            va="center",
        )
    )
adjust_text(texts)

# Draw a line connecting the current best model to the next best model
sorted_df = plot_df.sort_values(
    by=["release_date", "positive_dice"], ascending=[True, False]
)
current_best_model = sorted_df.iloc[0]
for i in range(1, len(sorted_df)):
    next_model = sorted_df.iloc[i]
    if next_model["positive_dice"] > current_best_model["positive_dice"]:
        ax.plot(
            [current_best_model["release_date"], next_model["release_date"]],
            [current_best_model["positive_dice"], next_model["positive_dice"]],
            color="gray",
            linestyle="--",
            linewidth=0.5,
        )
        current_best_model = next_model

ax.set_ylim(0.77, 0.84)
ax.set_ylabel("Mean Positive Dice Score")

x_labels = ax.get_xticklabels()
x_ticks = ax.get_xticks()
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels)
ax.set_xlabel("Release Date")

plt.tight_layout()
plt.savefig("segmentation_performance_by_release_date.pdf")
plt.savefig("segmentation_performance_by_release_date.png", dpi=300)
plt.show()

## Performance vs. Model Size

In [ ]:
from unittest.mock import patch

_BACKBONE_SPECS: list[tuple[str, type, dict]] = [
    (
        "deblurmim_convmae",
        DeblurMIMBackbone,
        {
            "checkpoint_path": "dummy",
            "output_intermediate_features": True,
            "model_variant": "deblurmim_convmae_base",
        },
    ),
    (
        "deblurmim_mae",
        DeblurMIMBackbone,
        {
            "checkpoint_path": "dummy",
            "output_intermediate_features": True,
            "model_variant": "deblurmim_mae_base_patch16",
        },
    ),
    (
        "dinov3_convnext_base",
        DINOv3ConvNeXtBackbone,
        {
            "checkpoint_path": "dummy",
            "repo_dir": "dummy",
            "output_intermediate_features": True,
            "model_variant": "dinov3_convnext_base",
        },
    ),
    (
        "dinov3_convnext_large",
        DINOv3ConvNeXtBackbone,
        {
            "checkpoint_path": "dummy",
            "repo_dir": "dummy",
            "output_intermediate_features": True,
            "model_variant": "dinov3_convnext_large",
        },
    ),
    (
        "dinov3_vitb16",
        DINOv3ViTBackbone,
        {
            "checkpoint_path": "dummy",
            "repo_dir": "dummy",
            "output_intermediate_features": True,
            "model_variant": "dinov3_vitb16",
        },
    ),
    (
        "dinov3_vitl16",
        DINOv3ViTBackbone,
        {
            "checkpoint_path": "dummy",
            "repo_dir": "dummy",
            "output_intermediate_features": True,
            "model_variant": "dinov3_vitl16",
        },
    ),
    (
        "echocare",
        EchoCareBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "openus",
        OpenUSBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "samus",
        SAMUSBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": False},
    ),
    (
        "tinyusfm",
        TinyUSFMBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "ultrafedfm",
        UltraFedFMBackbone,
        {
            "checkpoint_path": "dummy",
            "output_intermediate_features": True,
            "model_variant": "ultrafedfm_vit_base_patch16",
        },
    ),
    (
        "ultrasam",
        UltraSAMBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "urfm_base",
        URFMBackbone,
        {
            "checkpoint_path": "dummy",
            "model_variant": "urfm_mae_vit_base_patch16",
            "output_intermediate_features": True,
        },
    ),
    (
        "urfm_large",
        URFMBackbone,
        {
            "checkpoint_path": "dummy",
            "model_variant": "urfm_mae_vit_large_patch16",
            "output_intermediate_features": True,
        },
    ),
    (
        "usfm",
        USFMBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
    (
        "usfmae",
        USFMAEBackbone,
        {"checkpoint_path": "dummy", "output_intermediate_features": True},
    ),
]


model_parameter_counts = {}
for model_key, cls, kwargs in _BACKBONE_SPECS:
    if model_key == "dinov3_convnext_base":
        trainable_parameters = 89_000_000
    elif model_key == "dinov3_convnext_large":
        trainable_parameters = 198_000_000
    elif model_key == "dinov3_vitb16":
        trainable_parameters = 86_000_000
    elif model_key == "dinov3_vitl16":
        trainable_parameters = 300_000_000
    else:
        with patch.object(cls, "_load_checkpoint", lambda self: None):
            backbone = cls(**kwargs)

        # Count the number of trainable parameters in the model (excluding
        # "decoder" parameters)
        trainable_parameters = sum(
            p.numel()
            for name, p in backbone.named_parameters()
            if p.requires_grad and "decoder" not in name
        )

        del backbone

    model_parameter_counts[model_key] = trainable_parameters

In [ ]:
# Plot the overall classification performance on the X-axis, the overall
# segmentation performance on the Y-axis, and the number of trainable
# parameters (excluding decoder parameters) as the size of the points.
plot_df = (
    test_results[
        (test_results["shot_pct"] == 1.0)
        & (test_results["mode"].str.contains("full_finetuning"))
    ]
    .groupby(["model"])
    .agg(
        {
            "f1_macro": "mean",
            "positive_dice": "mean",
        }
    )
    .reset_index()
)
plot_df["params"] = plot_df["model"].map(model_parameter_counts)
plot_df["params"] = round(plot_df["params"] / 1e6, 1)  # Convert to millions

fig, ax = plt.subplots()

sns.scatterplot(
    data=plot_df,
    x="f1_macro",
    y="positive_dice",
    size="params",
    hue="params",
    ax=ax,
    legend=True,
)

# Add labels for each point
texts = []
for _, row in plot_df.iterrows():
    texts.append(
        ax.text(
            row["f1_macro"],
            row["positive_dice"],
            MODEL_DISPLAY_NAMES[row["model"]],
            ha="center",
            va="center",
        )
    )
adjust_text(texts)

handles, labels = ax.get_legend_handles_labels()
new_labels = [f"{float(label):.1f}M" for label in labels]
plt.legend(handles=handles, labels=new_labels, ncol=2)

ax.set_xlabel("Mean F1 Score")
ax.set_ylabel("Mean Positive Dice Score")

plt.tight_layout()
plt.savefig("performance_vs_model_size.pdf")
plt.savefig("performance_vs_model_size.png", dpi=300)
plt.show()

## Performance per Category

Report performance on pathological vs. non-pathological classification tasks and pathological vs. anatomical segmentation tasks.

In [ ]:
PATHO_SEG_TASKS = [
    "aul_mass_segmentation",
    "busbra_segmentation",
    "mmotu_segmentation",
    "tn3k_segmentation",
]
ANATOMICAL_SEG_TASKS = [
    "aul_liver_segmentation",
    "camus_segmentation",
    "open_kidney_capsule_segmentation",
    "open_kidney_regions_segmentation",
    "psfhs_segmentation",
    "stmus_segmentation",
]
PATHO_CLS_TASKS = [
    "aul_mass_classification",
    "busbra_histology_classification",
    "busbra_pathology_classification",
    "covid_blues_classification",
    "gbcu_classification",
    "masld_classification",
    "mmotu_classification",
    "tn3k_classification",
]
NONPATHO_CLS_TASKS = [
    "camus_classification",
    "fetal_planes_brain_plane_classification",
    "fetal_planes_plane_classification",
]

In [ ]:
# Plot full fine-tuning and linear probing classification performance for each
# group of tasks side-by-side using a dumbbell plot
def plot_classification_dumbbell(ax, plot_df, category, title):
    fft_df = (
        plot_df[
            (plot_df["category"] == category)
            & (plot_df["mode"].str.contains("full_finetuning"))
        ]
        .groupby("model")
        .agg({"macro_f1": "mean"})
        .reset_index()
        .sort_values(by="macro_f1", ascending=False)
    )
    lp_df = (
        plot_df[
            (plot_df["category"] == category)
            & (plot_df["mode"].str.contains("head_only"))
        ]
        .groupby("model")
        .agg({"macro_f1": "mean"})
        .reset_index()
    )

    # Use the fixed MODEL_COLORS mapping so each model gets the same color
    # here as in every other plot, regardless of this panel's sort order.
    sns.scatterplot(
        fft_df,
        x="model",
        y="macro_f1",
        ax=ax,
        hue="model",
        palette=MODEL_COLORS,
        legend=False,
    )
    sns.scatterplot(
        lp_df,
        x="model",
        y="macro_f1",
        ax=ax,
        hue="model",
        palette=MODEL_COLORS,
        legend=False,
        marker="D",
    )

    # Add vertical lines connecting the two points for each model
    for model in fft_df["model"]:
        fft_value = fft_df.loc[fft_df["model"] == model, "macro_f1"].values[0]
        lp_value = lp_df.loc[lp_df["model"] == model, "macro_f1"].values[0]
        ax.plot(
            [model, model],
            [fft_value, lp_value],
            color=MODEL_COLORS[model],
            linestyle="--",
            linewidth=0.5,
        )

    ax.set_title(title)
    ax.set_ylim(0.40, 0.75)

    x_ticks = [x + 0.5 for x in ax.get_xticks()]
    x_labels = [MODEL_DISPLAY_NAMES[x.get_text()] for x in ax.get_xticklabels()]
    ax.set_xticks(x_ticks, x_labels, rotation=45, ha="right")
    ax.set_xlabel("")


fig, axes = plt.subplots(1, 2, sharey=True)

plot_df = classification_test_summary[
    classification_test_summary["shot_pct"] == 1.0
].copy()
plot_df["category"] = plot_df["task"].map(
    lambda x: "pathological" if x in PATHO_CLS_TASKS else "non-pathological"
)

plot_classification_dumbbell(axes[0], plot_df, "pathological", "Pathological")
axes[0].set_ylabel("Macro F1")

plot_classification_dumbbell(axes[1], plot_df, "non-pathological", "Non-Pathological")

plt.tight_layout()
plt.savefig("performance_per_category_classification.pdf")
plt.savefig("performance_per_category_classification.png", dpi=300)
plt.show()

In [ ]:
# Plot full fine-tuning and linear probing segmentation performance for each
# group of tasks side-by-side using a dumbbell plot
def plot_segmentation_dumbbell(ax, plot_df, category, title):
    fft_df = (
        plot_df[
            (plot_df["category"] == category)
            & (plot_df["mode"].str.contains("full_finetuning"))
        ]
        .groupby("model")
        .agg({"positive_dice": "mean"})
        .reset_index()
        .sort_values(by="positive_dice", ascending=False)
    )
    lp_df = (
        plot_df[
            (plot_df["category"] == category)
            & (plot_df["mode"].str.contains("head_only"))
        ]
        .groupby("model")
        .agg({"positive_dice": "mean"})
        .reset_index()
    )

    # Use the fixed MODEL_COLORS mapping so each model gets the same color
    # here as in every other plot, regardless of this panel's sort order.
    sns.scatterplot(
        fft_df,
        x="model",
        y="positive_dice",
        ax=ax,
        hue="model",
        palette=MODEL_COLORS,
        legend=False,
    )
    sns.scatterplot(
        lp_df,
        x="model",
        y="positive_dice",
        ax=ax,
        hue="model",
        palette=MODEL_COLORS,
        legend=False,
        marker="D",
    )

    # Add vertical lines connecting the two points for each model
    for model in fft_df["model"]:
        fft_value = fft_df.loc[fft_df["model"] == model, "positive_dice"].values[0]
        lp_value = lp_df.loc[lp_df["model"] == model, "positive_dice"].values[0]
        ax.plot(
            [model, model],
            [fft_value, lp_value],
            color=MODEL_COLORS[model],
            linestyle="--",
            linewidth=0.5,
        )

    ax.set_title(title)
    ax.set_ylim(0.55, 0.85)

    x_ticks = [x + 0.5 for x in ax.get_xticks()]
    x_labels = [MODEL_DISPLAY_NAMES[x.get_text()] for x in ax.get_xticklabels()]
    ax.set_xticks(x_ticks, x_labels, rotation=45, ha="right")
    ax.set_xlabel("")


fig, axes = plt.subplots(1, 2, sharey=True)

plot_df = segmentation_test_summary[segmentation_test_summary["shot_pct"] == 1.0].copy()
plot_df["category"] = plot_df["task"].map(
    lambda x: "pathological" if x in PATHO_SEG_TASKS else "anatomical"
)

plot_segmentation_dumbbell(axes[0], plot_df, "pathological", "Pathological")
axes[0].set_ylabel("Positive Dice")

plot_segmentation_dumbbell(axes[1], plot_df, "anatomical", "Anatomical")

plt.tight_layout()
plt.savefig("performance_per_category_segmentation.pdf")
plt.savefig("performance_per_category_segmentation.png", dpi=300)
plt.show()

## Performance per Task

In [ ]:
# Short task labels, used as subplot titles so that the panels fit the page.
TASK_DISPLAY_NAMES = {
    # Classification
    "aul_mass_classification": "AUL Mass",
    "busbra_histology_classification": "BUS-BRA Hist.",
    "busbra_pathology_classification": "BUS-BRA Path.",
    "camus_classification": "CAMUS",
    "covid_blues_classification": "COVID-BLUES",
    "fetal_planes_brain_plane_classification": "Fetal Brain",
    "fetal_planes_plane_classification": "Fetal Plane",
    "gbcu_classification": "GBCU",
    "masld_classification": "MASLD",
    "mmotu_classification": "MMOTU",
    "tn3k_classification": "TN3K",
    # Segmentation
    "aul_liver_segmentation": "AUL Liver",
    "aul_mass_segmentation": "AUL Mass",
    "busbra_segmentation": "BUS-BRA",
    "camus_segmentation": "CAMUS",
    "mmotu_segmentation": "MMOTU",
    "open_kidney_capsule_segmentation": "OK Capsule",
    "open_kidney_regions_segmentation": "OK Regions",
    "psfhs_segmentation": "PSFHS",
    "stmus_segmentation": "STMUS",
    "tn3k_segmentation": "TN3K",
}

# Font scale used for the per-task grids, found by trial and error to keep 15
# rotated model names on the X-axis legible at this panel size.
TASK_FONT_SCALE = 0.6

# Number of Y-axis ticks to use in every panel, so that panels with different
# autoscaled ranges still look consistent side-by-side.
TASK_N_YTICKS = 5


def summarize_per_task(results: pd.DataFrame, metric: str) -> pd.DataFrame:
    """Summarize a metric over the random seeds of each task and model.

    The decoder prefix is stripped from `mode` so that the results are grouped
    by training paradigm alone (the segmentation decoder depends on the
    backbone architecture).

    Args:
        results: The unaggregated test results for a single type of task.
        metric: The name of the metric column to summarize.

    Returns:
        A data frame with one row per task, model and paradigm, and `mean` and
        `stderr` columns holding the summary statistics over the seeds.
    """
    summary = results[results["shot_pct"] == 1.0].copy()
    summary["paradigm"] = summary["mode"].str.replace(
        r"^.*?_(full_finetuning|head_only)$", r"\1", regex=True
    )
    summary = (
        summary.groupby(["task", "model", "paradigm"])[metric]
        .agg(["mean", "sem"])
        .reset_index()
    )

    return summary.rename(columns={"sem": "stderr"})


def plot_task_performance(ax, plot_df: pd.DataFrame, task: str) -> None:
    """Plot the mean and std. err. of each model on a single task.

    The models are ordered along the X-axis in descending order of their mean
    performance, and the Y-axis is left to autoscale to the task.

    Args:
        ax: The axes to draw the panel on.
        plot_df: A summary with `task`, `model`, `mean` and `stderr` columns.
        task: The name of the task to plot.
    """
    task_df = plot_df[plot_df["task"] == task].sort_values(by="mean", ascending=False)

    # Use the fixed MODEL_COLORS mapping so each model gets the same color here
    # as in every other plot, regardless of this panel's sort order.
    for position, (model, mean, stderr) in enumerate(
        zip(task_df["model"], task_df["mean"], task_df["stderr"])
    ):
        ax.errorbar(
            position,
            mean,
            yerr=stderr,
            color=MODEL_COLORS[model],
            marker="o",
            markersize=3,
            elinewidth=0.8,
            capsize=1.5,
        )

    ax.set_title(TASK_DISPLAY_NAMES[task])
    ax.set_xlim(-0.5, len(task_df) - 0.5)
    ax.set_xticks(
        range(len(task_df)),
        [MODEL_DISPLAY_NAMES[model] for model in task_df["model"]],
        rotation=90,
    )
    ax.set_xlabel("")

    y_min, y_max = ax.get_ylim()
    ax.set_yticks(np.linspace(y_min, y_max, TASK_N_YTICKS))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))


classification_task_summary = summarize_per_task(
    classification_test_results, "f1_macro"
)
segmentation_task_summary = summarize_per_task(
    segmentation_test_results, "positive_dice"
)
segmentation_task_summary.head()

In [ ]:
# Plot the head-only classification performance of each model on each task,
# with the models ordered by mean performance and error bars showing the std.
# err. over the seeds
tasks = PATHO_CLS_TASKS + NONPATHO_CLS_TASKS
plot_df = classification_task_summary[
    classification_task_summary["paradigm"] == "head_only"
]

with sns.plotting_context("paper", font_scale=TASK_FONT_SCALE):
    fig, axes = plt.subplots(3, 4, figsize=(6.5, 8.0))

    for ax, task in zip(axes.flat, tasks):
        plot_task_performance(ax, plot_df, task)

    for ax in axes.flat[len(tasks) :]:
        ax.set_visible(False)

    for ax in axes[:, 0]:
        ax.set_ylabel("Macro F1")

    plt.tight_layout()
    plt.savefig("performance_per_task_classification_head_only.pdf")
    plt.savefig("performance_per_task_classification_head_only.png", dpi=300)
    plt.show()

In [ ]:
# Plot the full fine-tuning classification performance of each model on each
# task, with the models ordered by mean performance and error bars showing the
# std. err. over the seeds
tasks = PATHO_CLS_TASKS + NONPATHO_CLS_TASKS
plot_df = classification_task_summary[
    classification_task_summary["paradigm"] == "full_finetuning"
]

with sns.plotting_context("paper", font_scale=TASK_FONT_SCALE):
    fig, axes = plt.subplots(3, 4, figsize=(6.5, 8.0))

    for ax, task in zip(axes.flat, tasks):
        plot_task_performance(ax, plot_df, task)

    for ax in axes.flat[len(tasks) :]:
        ax.set_visible(False)

    for ax in axes[:, 0]:
        ax.set_ylabel("Macro F1")

    plt.tight_layout()
    plt.savefig("performance_per_task_classification_full_finetuning.pdf")
    plt.savefig("performance_per_task_classification_full_finetuning.png", dpi=300)
    plt.show()

In [ ]:
# Plot the head-only segmentation performance of each model on each task, with
# the models ordered by mean performance and error bars showing the std. err.
# over the seeds
tasks = PATHO_SEG_TASKS + ANATOMICAL_SEG_TASKS
plot_df = segmentation_task_summary[
    segmentation_task_summary["paradigm"] == "head_only"
]

with sns.plotting_context("paper", font_scale=TASK_FONT_SCALE):
    fig, axes = plt.subplots(3, 4, figsize=(6.5, 8.0))

    for ax, task in zip(axes.flat, tasks):
        plot_task_performance(ax, plot_df, task)

    for ax in axes.flat[len(tasks) :]:
        ax.set_visible(False)

    for ax in axes[:, 0]:
        ax.set_ylabel("Positive Dice")

    plt.tight_layout()
    plt.savefig("performance_per_task_segmentation_head_only.pdf")
    plt.savefig("performance_per_task_segmentation_head_only.png", dpi=300)
    plt.show()

In [ ]:
# Plot the full fine-tuning segmentation performance of each model on each
# task, with the models ordered by mean performance and error bars showing the
# std. err. over the seeds
tasks = PATHO_SEG_TASKS + ANATOMICAL_SEG_TASKS
plot_df = segmentation_task_summary[
    segmentation_task_summary["paradigm"] == "full_finetuning"
]

with sns.plotting_context("paper", font_scale=TASK_FONT_SCALE):
    fig, axes = plt.subplots(3, 4, figsize=(6.5, 8.0))

    for ax, task in zip(axes.flat, tasks):
        plot_task_performance(ax, plot_df, task)

    for ax in axes.flat[len(tasks) :]:
        ax.set_visible(False)

    for ax in axes[:, 0]:
        ax.set_ylabel("Positive Dice")

    plt.tight_layout()
    plt.savefig("performance_per_task_segmentation_full_finetuning.pdf")
    plt.savefig("performance_per_task_segmentation_full_finetuning.png", dpi=300)
    plt.show()